# Runnable Protocol and LCEL

Every LangChain component — prompts, chat models, output parsers, retrievers, tools — implements the same interface: **`Runnable`**. That single contract is what makes LangChain composable. You chain steps with the pipe operator **`|`** using **LCEL** (LangChain Expression Language), and the resulting pipeline automatically supports synchronous calls, batching, streaming, async, callbacks, and retries.

**01. Introduction to LangChain** previewed Runnables and LCEL with a minimal example. This notebook goes deep: the full execution API, how piping works internally, wrapping plain Python functions, passing dicts through chains, `RunnableConfig`, and the composition primitives (`RunnablePassthrough`, `RunnableParallel`) that every RAG and agent pipeline relies on.

Advanced branching, complex dict routing, and production resilience patterns are extended in **06. LCEL Composition Patterns** and **16. Fallbacks and Configurable Runnables**. Chat models, prompts, and parsers get their own notebooks (**03–05**); here we use them only where needed to demonstrate Runnable behavior.

By the end you will be able to read, write, and debug any LangChain chain — with or without an LLM in the loop.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json
from typing import Iterator

import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain-core:", langchain_core.__version__)

---

## 1. What Is the Runnable Protocol?

A **Runnable** is LangChain's abstraction for a unit of work: something that accepts an input, performs a transformation, and returns an output. The protocol defines **how** that work is executed — not **what** the work is.

```
Input ──► Runnable ──► Output
              │
              ├── invoke()   single call
              ├── batch()    many calls
              ├── stream()   incremental output
              └── ainvoke()  async single call
```

Because prompts, LLMs, and parsers all implement Runnable, you compose them identically:

$$\text{chain} = r_1 \mid r_2 \mid r_3 \quad \Rightarrow \quad \text{chain.invoke}(x) = r_3\big(r_2(r_1(x))\big)$$

The protocol lives in **`langchain-core`**. Provider packages add Runnable implementations; your application code composes them.

---

## 2. The Runnable Execution API

| Method | Sync / Async | Input | Output | Use case |
|--------|--------------|-------|--------|----------|
| `invoke(input, config=None)` | Sync | One item | One result | API request handler |
| `batch(inputs, config=None)` | Sync | List | List | Bulk processing |
| `stream(input, config=None)` | Sync | One item | Iterator of chunks | UX token streaming |
| `ainvoke(input, config=None)` | Async | One item | Awaitable | FastAPI async routes |
| `abatch(inputs, config=None)` | Async | List | Awaitable list | Async bulk |
| `astream(input, config=None)` | Async | One item | Async iterator | Async streaming |

Every method accepts an optional **`RunnableConfig`** — tags, metadata, callbacks, and configurable fields passed through the entire chain. Section 10 covers config in detail.

**Streaming semantics:** not every Runnable emits multiple chunks. A `RunnableLambda` yields one chunk (the full output). Chat models yield token chunks. A chain's `stream()` propagates whatever each step produces according to LangChain's streaming rules.

---

## 3. RunnableLambda — Wrap Any Function

**`RunnableLambda`** turns a Python callable into a Runnable. Use it for custom logic between LangChain components — formatting documents, filtering results, calling external APIs.

The function can accept:

- A plain value (`str`, `int`, …)
- A **dict** when the chain passes structured state
- **`RunnableConfig`** as a second argument if you declare `config: RunnableConfig` in the signature

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableConfig


def slugify(text: str) -> str:
    return text.lower().replace(" ", "-")


def with_meta(text: str, config: RunnableConfig) -> dict:
    tags = config.get("tags", [])
    return {"slug": slugify(text), "tags": tags}


r_slug = RunnableLambda(slugify)
r_meta = RunnableLambda(with_meta)

print("invoke:", r_slug.invoke("Hello World"))
print("batch:", r_slug.batch(["Fast API", "JWT Auth"]))
print("with config:", r_meta.invoke("Hello World", config=RunnableConfig(tags=["demo"])))
print("is Runnable:", hasattr(r_slug, "invoke"), hasattr(r_slug, "stream"))

### 3.1 Sync vs Async Lambdas

If the wrapped function is `async def`, LangChain routes `ainvoke` / `astream` to the async path automatically:

In [ ]:
async def async_double(x: int) -> int:
    return x * 2


async_r = RunnableLambda(async_double)
print("ainvoke:", asyncio.run(async_r.ainvoke(21)))

---

## 4. LCEL — The Pipe Operator

**LCEL** (LangChain Expression Language) is the syntax for composing Runnables with **`|`**:

```python
chain = step_a | step_b | step_c
result = chain.invoke(input)
```

Equivalent imperative code:

```python
def chain_invoke(x):
    x = step_a.invoke(x)
    x = step_b.invoke(x)
    x = step_c.invoke(x)
    return x
```

The piped object is a **`RunnableSequence`**. It inherits the full Runnable API — `invoke` on the sequence calls each step in order.

In [ ]:
def add_exclaim(text: str) -> str:
    return text + "!"


def wrap_brackets(text: str) -> str:
    return f"[{text}]"


chain = RunnableLambda(add_exclaim) | RunnableLambda(wrap_brackets)

print("type:", type(chain).__name__)
print("steps:", [type(s).__name__ for s in chain.steps])
print("result:", chain.invoke("hello"))
print("batch:", chain.batch(["a", "b"]))

### 4.1 Chaining Dict Outputs to Dict Inputs

When step A returns a **dict** and step B expects specific keys, LangChain passes the dict through. If step B is a prompt template, it reads keys as template variables:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate


def build_context(data: dict) -> dict:
    return {
        "question": data["question"],
        "context": f"Source: {data['source']} — {data['snippet']}",
    }


template = ChatPromptTemplate.from_messages([
    ("system", "Answer using context only.\n{context}"),
    ("human", "{question}"),
])

dict_chain = RunnableLambda(build_context) | template

prompt_value = dict_chain.invoke({
    "question": "What header is used?",
    "source": "security-docs",
    "snippet": "Authorization: Bearer token",
})

for m in prompt_value.messages:
    print(f"[{m.type}] {m.content[:80]}..." if len(m.content) > 80 else f"[{m.type}] {m.content}")

---

## 5. invoke — Single Execution

`invoke` is the workhorse for server request handlers. One input, one output, blocking until complete.

| Input type | Typical first step |
|------------|-------------------|
| `str` | Simple lambda or prompt with one variable |
| `dict` | Prompt template with multiple variables |
| `list[Message]` | Direct chat model call (see **03**) |

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer in at most 10 words."),
    ("human", "{question}"),
])

qa_chain = qa_prompt | llm | StrOutputParser()

print(qa_chain.invoke({"question": "What is an API?"}))
print("return type:", type(qa_chain.invoke({"question": "What is JSON?"})).__name__)

---

## 6. batch — Parallel Processing

`batch` accepts a list of inputs and returns a list of outputs in the same order. LangChain may parallelize independent work (provider rate limits still apply).

Use `batch` for offline eval, bulk summarization, or processing a queue of user questions.

In [ ]:
questions = [
    {"question": "What is HTTP?"},
    {"question": "What is REST?"},
    {"question": "What is JSON?"},
]

answers = qa_chain.batch(questions)
for q, a in zip(questions, answers):
    print(f"Q: {q['question']}")
    print(f"A: {a}\n")

### 6.1 batch_size and max_concurrency

Pass via config to control concurrency:

```python
chain.batch(inputs, config=RunnableConfig(max_concurrency=5))
```

Tune this against provider rate limits. **07. Streaming, Batch, and Async** explores batch vs async trade-offs in production.

---

## 7. stream — Incremental Output

`stream` yields chunks as they become available. For `StrOutputParser` on a chat model, chunks are **text tokens** as the model generates.

In [ ]:
stream_prompt = ChatPromptTemplate.from_template("List three benefits of {topic} as bullets.")
stream_chain = stream_prompt | llm | StrOutputParser()

print("Streaming tokens:")
chunks: list[str] = []
for chunk in stream_chain.stream({"topic": "automated testing"}):
    chunks.append(chunk)
    print(chunk, end="", flush=True)

print("\n---")
print("chunk count:", len(chunks))
print("reconstructed:", "".join(chunks)[:100], "...")

### 7.1 stream vs invoke

| | `invoke` | `stream` |
|---|---------|----------|
| **Returns** | Final output only | Iterator of partial outputs |
| **Latency perception** | Waits for completion | First token sooner |
| **Use in UI** | Simple APIs | Chat interfaces, SSE |
| **Total cost** | Same tokens | Same tokens |

Streaming does not reduce token cost — it improves **time-to-first-token** for the user.

---

## 8. Async — ainvoke, abatch, astream

Async methods integrate with FastAPI, asyncio servers, and concurrent I/O-bound LLM calls.

In [ ]:
async def demo_async():
    result = await qa_chain.ainvoke({"question": "What is async?"})
    print("ainvoke:", result)

    batch_results = await qa_chain.abatch([
        {"question": "What is await?"},
        {"question": "What is asyncio?"},
    ])
    print("abatch:", batch_results)

    print("astream:", end=" ")
    async for chunk in qa_chain.astream({"question": "Say hi in 3 words"}):
        print(chunk, end="", flush=True)
    print()


asyncio.run(demo_async())

---

## 9. RunnablePassthrough — Forward and Enrich Input

**`RunnablePassthrough`** passes input through unchanged — or **merges** new keys into a dict with `.assign()`.

This is essential for RAG: keep the original question while adding retrieved context.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Pass through unchanged
print("passthrough:", RunnablePassthrough().invoke("unchanged"))

# assign() adds keys to a dict input
enricher = RunnablePassthrough.assign(
    doubled=RunnableLambda(lambda x: x["value"] * 2),
    label=RunnableLambda(lambda x: f"item-{x['value']}"),
)

print("assign:", enricher.invoke({"value": 7, "name": "demo"}))

### 9.1 The RAG Dict Pattern (Preview)

Canonical RAG uses `RunnableParallel` + passthrough — fully built in **11. RAG with LCEL**:

In [ ]:
FAKE_CORPUS = {
    "jwt": "JWT bearer tokens use Authorization: Bearer header.",
    "alembic": "Run alembic upgrade head to apply migrations.",
}


def fake_retrieve(question: str) -> str:
    q = question.lower()
    if "jwt" in q or "auth" in q:
        return FAKE_CORPUS["jwt"]
    if "alembic" in q or "migrat" in q:
        return FAKE_CORPUS["alembic"]
    return "No docs found."


rag_dict_chain = RunnableParallel(
    context=RunnableLambda(fake_retrieve),
    question=RunnablePassthrough(),
)

print(rag_dict_chain.invoke("How does JWT auth work?"))
print(rag_dict_chain.invoke("Alembic migration command?"))

---

## 10. RunnableParallel — Fan-Out

**`RunnableParallel`** runs multiple Runnables against the **same input** and returns a **dict** of results. Keys are the names you assign.

```
         ┌──► branch_a ──► result_a ──┐
input ───┤                            ├──► {a: ..., b: ...}
         └──► branch_b ──► result_b ──┘
```

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    original=RunnablePassthrough(),
    upper=RunnableLambda(lambda x: x.upper()),
    length=RunnableLambda(lambda x: len(x)),
    words=RunnableLambda(lambda x: len(x.split())),
)

print(parallel.invoke("hello runnable world"))
print(parallel.batch(["one", "two words"]))

### 10.1 Shorthand Dict Syntax

A dict of Runnables in a sequence step is automatically treated as RunnableParallel:

In [ ]:
# Equivalent: {"a": r1, "b": r2} inside a chain creates RunnableParallel
shorthand_chain = {
    "text": RunnablePassthrough(),
    "hash": RunnableLambda(lambda x: hash(x) % 10000),
} | RunnableLambda(lambda d: f"{d['text']} (#{d['hash']})")

print(shorthand_chain.invoke("langchain"))

---

## 11. RunnableConfig — Cross-Cutting Context

**`RunnableConfig`** carries metadata through every step in a chain invocation:

| Field | Purpose |
|-------|--------|
| `run_name` | Label in traces and logs |
| `tags` | Filter/group runs in LangSmith |
| `metadata` | Arbitrary key-value for observability |
| `callbacks` | Handlers for lifecycle events |
| `configurable` | Runtime-swappable fields (see **16**) |
| `max_concurrency` | Batch parallelism limit |

In [ ]:
config = RunnableConfig(
    run_name="qa_demo",
    tags=["lesson-02", "runnable"],
    metadata={"notebook": "02-runnable-lcel"},
)

answer = qa_chain.invoke({"question": "What is LCEL?"}, config=config)
print(answer)

---

## 12. Binding and Partial Application

### 12.1 bind() — Attach Runtime Options

`.bind()` fixes kwargs on a Runnable — commonly used to attach tools to a chat model:

In [ ]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


llm_with_tools = llm.bind_tools([add])
response = llm_with_tools.invoke("Use the add tool for 40 + 2")
print("tool_calls:", response.tool_calls)
print("content:", response.content[:80] if response.content else "(empty — model chose tool)")

### 12.2 partial() — Pre-Fill Template Variables

Prompt templates support `.partial()` to freeze variables that do not change per request:

In [ ]:
flex_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Reply in {language}."),
    ("human", "{question}"),
])

teacher_prompt = flex_prompt.partial(role="patient teacher", language="English")

teacher_chain = teacher_prompt | llm | StrOutputParser()
print(teacher_chain.invoke({"question": "What is a Runnable?"}))

---

## 13. Chain Inspection and Debugging

Understand what you built before tracing in LangSmith.

In [ ]:
full_chain = (
    RunnableParallel(context=RunnableLambda(fake_retrieve), question=RunnablePassthrough())
    | ChatPromptTemplate.from_messages([
        ("system", "Context: {context}"),
        ("human", "{question}"),
    ])
    | llm
    | StrOutputParser()
)

print("Chain type:", type(full_chain).__name__)
print("Steps:", [type(s).__name__ for s in full_chain.steps])
print()
print("Answer:", full_chain.invoke("Explain JWT authentication briefly."))

### 13.1 Pick Intermediate Outputs with assign

Debug by capturing intermediate dict keys before the final LLM call:

In [ ]:
debug_chain = (
    RunnableParallel(context=RunnableLambda(fake_retrieve), question=RunnablePassthrough())
    | RunnablePassthrough.assign(
        answer=(
            ChatPromptTemplate.from_messages([
                ("system", "Context: {context}"),
                ("human", "{question}"),
            ])
            | llm
            | StrOutputParser()
        )
    )
)

debug_out = debug_chain.invoke("Alembic upgrade command?")
print(json.dumps(debug_out, indent=2))

---

## 14. Resilience Hooks (Preview)

Runnables support `.with_retry()` and `.with_fallbacks()`. Full patterns are in **16. Fallbacks and Configurable Runnables**.

In [ ]:
def flaky(text: str) -> str:
    if "error" in text.lower():
        raise RuntimeError("simulated provider error")
    return text.upper()


robust = (
    RunnableLambda(flaky)
    .with_retry(stop_after_attempt=2, wait_exponential_jitter=True)
    .with_fallbacks([RunnableLambda(lambda x: f"FALLBACK({x})")])
)

print(robust.invoke("ok"))
print(robust.invoke("trigger error"))

---

## 15. LCEL vs Legacy Chains

| Aspect | Legacy `LLMChain` | LCEL |
|--------|-------------------|------|
| Syntax | Class + `.run()` | `prompt \| llm \| parser` |
| Streaming | Extra setup | Native `chain.stream()` |
| Async | Separate code path | Native `chain.ainvoke()` |
| Composition | Nested chain objects | Pipe operator |
| Debugging | Opaque | Inspect `.steps` |

New code should use LCEL exclusively. Legacy chains remain in old tutorials and some `langchain-community` recipes.

---

## 16. Runnable Type Flow — Mental Model

Track types through a chain to avoid surprises:

```
dict ──► ChatPromptTemplate ──► ChatPromptValue (messages)
                                      │
                                      ▼
                                 ChatOpenAI ──► AIMessage
                                      │
                                      ▼
                              StrOutputParser ──► str
```

| Step | Typical input | Typical output |
|------|---------------|----------------|
| `ChatPromptTemplate` | `dict` of variables | `ChatPromptValue` |
| `ChatOpenAI` | messages or prompt value | `AIMessage` |
| `StrOutputParser` | `AIMessage` or `str` | `str` |
| `RunnableLambda` | whatever you define | whatever you return |
| `RunnableParallel` | any | `dict` of branch outputs |

In [ ]:
# Visualize Runnable methods available on every chain
methods = ["invoke", "batch", "stream", "ainvoke", "abatch", "astream"]
available = [m for m in methods if hasattr(qa_chain, m)]

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(available, [1] * len(available), color="#4c72b0")
ax.set_xlim(0, 1.5)
ax.set_title("Runnable API methods on a composed LCEL chain")
ax.set_xlabel("available")
plt.tight_layout()
plt.show()

---

## 17. Design Patterns with LCEL

| Pattern | LCEL shape | Notebook |
|---------|------------|----------|
| **Linear transform** | `a \| b \| c` | This notebook |
| **Enrich dict** | `RunnablePassthrough.assign(...)` | This notebook |
| **Fan-out** | `RunnableParallel(...)` | This notebook, **06** |
| **Conditional route** | `RunnableBranch(...)` | **06** |
| **RAG** | `Parallel(context, question) \| prompt \| llm` | **11** |
| **Tool call** | `llm.bind_tools(...)` | **12** |
| **Multi-turn** | `RunnableWithMessageHistory` | **14** |

---

## 18. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Missing parser after LLM | `AIMessage` instead of `str` | Add `StrOutputParser()` |
| Wrong dict keys into prompt | KeyError or empty variables | Match template `{vars}` to dict keys |
| `RunnablePassthrough()` on a string when you need dict | Type errors downstream | Wrap or use `assign` on dict input |
| Expecting parallel branches to see each other's output | Branches are isolated | Add a merge step after `Parallel` |
| Ignoring `config` in custom lambdas | No tracing metadata | Accept `config: RunnableConfig` |
| Using `stream` on non-streaming steps only | Single chunk | Normal — still valid for uniform API |

---

## 19. Summary

The **Runnable protocol** is LangChain's foundation: `invoke`, `batch`, `stream`, and async variants on every component. **LCEL** composes Runnables with `|`, producing a `RunnableSequence` that inherits the full API.

Key takeaways:

- **`RunnableLambda`** wraps any function; use it for custom transforms.
- **`RunnablePassthrough`** forwards input; `.assign()` enriches dicts — critical for RAG.
- **`RunnableParallel`** fan-out produces named dict branches from one input.
- **`RunnableConfig`** carries tags, metadata, and callbacks through the chain.
- **`.bind()`** and **`.partial()`** attach tools and freeze prompt variables.
- **`.with_retry()` / `.with_fallbacks()`** add resilience (detailed in **16**).

Demonstrations covered lambdas, piping, dict chains, live LLM invoke/batch/stream/async, parallel/passthrough RAG dicts, bind_tools, debug assign, and retry/fallback previews.

Next: **03. Chat Models and Messages** — message types, model parameters, provider switching, and everything that happens inside the LLM step of your chains.